# W7C2 Lab: Masked Language Modelling and Transfer

Run every cell from the top. **Everything already works.**

Uses distilbert (about 260 MB), downloaded once and then cached.

Today you will:

1. Watch BERT fill in a blank, and see what it considers.
2. Find a sentence where its guess reveals a bias.
3. Reuse a frozen BERT as features for your own classifier.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel, logging

logging.set_verbosity_error()      # hide the load report; it is expected here

NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(NAME)
mlm = AutoModelForMaskedLM.from_pretrained(NAME)
mlm.eval()

MASK = tokenizer.mask_token
print("mask token:", MASK)
print(f"{sum(p.numel() for p in mlm.parameters()):,} parameters")

## Part 1. Filling in the blank

BERT was trained by hiding words and guessing them. That is the whole
pretraining task, and the model will happily still do it for you.

In [ ]:
# GIVEN. Top predictions for a masked position.
@torch.no_grad()
def fill(sentence, k=6):
    """sentence must contain exactly one [MASK]."""
    ids = tokenizer(sentence, return_tensors="pt")
    logits = mlm(**ids).logits
    spot = (ids["input_ids"][0] == tokenizer.mask_token_id).nonzero()[0, 0]
    probs = logits[0, spot].softmax(dim=-1)
    top = probs.topk(k)
    return pd.DataFrame({"word": [tokenizer.decode([i]) for i in top.indices],
                         "probability": top.values.numpy().round(3)})

s = f"the capital of france is {MASK} ."
print(s)
print(fill(s).to_string(index=False))
print()
print("Read that carefully. It is WRONG: it puts marseille first and paris fourth.")
print("Notice what it DID get right: every suggestion is a French city. The model")
print("learned which words keep company with which, not a fact about France.")

In [ ]:
# GIVEN. The same thing, drawn, for a sentence about weather.
s = f"it was raining so i took my {MASK} ."
guesses = fill(s, k=8)
print(s)
plt.figure(figsize=(7, 3))
plt.bar(guesses["word"], guesses["probability"], color="#7C2529")
plt.ylabel("probability"); plt.xticks(rotation=45, ha="right")
plt.title("What goes in the blank?"); plt.tight_layout(); plt.show()
print(guesses.to_string(index=False))

In [ ]:
# ================== YOUR TURN 1 ==================
# Write your own sentence with exactly one blank and see what BERT
# suggests. Use the MASK variable, and keep a space either side of it.
#
# Then try one designed to expose a stereotype, for example:
#    the nurse said [MASK] would be back shortly
#    the engineer said [MASK] would be back shortly
#
# Expected: the job sentences return DIFFERENT pronouns at different
#           confidences. Nothing in this code chose that; it came from the text
#           distilbert was trained on, and it is the Week 4 embedding-bias result
#           showing up in a much bigger model. Note the probabilities are low
#           throughout: a small model is rarely confident about anything.
# ===============================================
MY_SENTENCE = f"the nurse said {MASK} would be back shortly ."   # <-- change me

print(MY_SENTENCE)
print(fill(MY_SENTENCE).to_string(index=False))

## Part 2. Transfer: reuse the model, train almost nothing

The expensive part is already done. Freeze BERT, use its sentence vectors
as features, and fit a tiny classifier on top. That is transfer learning,
and on a small dataset it beats training from scratch every time.

In [ ]:
# GIVEN. Frozen BERT features + logistic regression.
import random
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

random.seed(0)
GOOD = ["brilliant", "great", "moving", "superb", "wonderful", "clever"]
BAD = ["dull", "boring", "awful", "weak", "terrible", "lazy"]
NOUNS = ["film", "script", "cast", "acting", "story", "ending"]
texts, labels = [], []
for _ in range(160):
    positive = random.random() < 0.5
    w = random.choice(GOOD if positive else BAD)
    texts.append(f"the {random.choice(NOUNS)} was {w}")
    labels.append(int(positive))

encoder = AutoModel.from_pretrained(NAME); encoder.eval()

@torch.no_grad()
def embed(batch):
    ids = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
    return encoder(**ids).last_hidden_state.mean(dim=1).numpy()

features = np.vstack([embed(texts[i:i + 32]) for i in range(0, len(texts), 32)])
print("BERT features:", features.shape, " <- 768 numbers per review")

Xtr, Xte, ytr, yte = train_test_split(features, labels, test_size=0.3,
                                      random_state=0, stratify=labels)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"frozen BERT + logistic regression: {accuracy_score(yte, clf.predict(Xte)):.2f}")

In [ ]:
# ================== YOUR TURN 2 ==================
# How much data does transfer learning actually need? Shrink the
# training set and see when it falls over.
#
# Try N_TRAIN = 8, then 20, then 100.
#
# Expected: even 8 labelled reviews gets a good score, because BERT already knows
#           what the words mean and the classifier only has to find a boundary.
#           Compare that against Week 2, where Naive Bayes needed hundreds. The
#           cell prints both so you can see the gap.
# ===============================================
N_TRAIN = 20          # <-- try 8, then 100

idx = np.arange(len(Xtr))[:N_TRAIN]
small = LogisticRegression(max_iter=1000).fit(Xtr[idx], np.array(ytr)[idx])
bert_acc = accuracy_score(yte, small.predict(Xte))

# The same amount of data, with Week 2's method and no pretrained model.
raw_train = [texts[i] for i in range(N_TRAIN)]
raw_y = labels[:N_TRAIN]
if len(set(raw_y)) > 1:
    cv = CountVectorizer()
    nb = MultinomialNB().fit(cv.fit_transform(raw_train), raw_y)
    nb_acc = accuracy_score(yte, nb.predict(cv.transform([texts[i] for i in range(len(texts))])[:len(yte)]))
else:
    nb_acc = float("nan")

print(f"training reviews: {N_TRAIN}")
print(f"   frozen BERT + logistic regression : {bert_acc:.2f}")
print(f"   bag of words + Naive Bayes        : {nb_acc:.2f}")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   The nurse/engineer pair returns different pronouns. No line of this
#   notebook encodes that; it is in the pretraining data, and it is the same
#   finding as the Week 4 embedding-bias task on a much bigger model.
#
#   Worth dwelling on: distilbert also gets 'the capital of france' WRONG,
#   ranking marseille above paris. Masked language modelling teaches a model
#   which words go together, which is not the same as teaching it facts.
#
# YOUR TURN 2
#   With 8 examples the frozen-BERT classifier is already usable, because the
#   768 numbers per sentence already encode meaning and only the decision
#   boundary is being learned. That is the entire argument for pretraining:
#   somebody else paid for the expensive part.